# P5 follow-ups on Gemma 3 12B (preregistered: `docs/preregistration_p5.md`)

**Before you press Run all (once):**
1. Upload `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`**.
2. Add your Hugging Face token as a Colab secret named **`HF_TOKEN`** (key icon on the left), and allow notebook access.
   The token's account must have accepted the Gemma 3 licence on huggingface.co.
3. **Runtime → Change runtime type → A100 GPU.** (12B in bf16 needs ~24 GB; an L4 is too tight.)

Then **Runtime → Run all** and keep the tab open. Expected time on an A100: about 1.5–2 hours.
Results are copied to `MyDrive/geomap/p5_results.zip` after each experiment, so a disconnect loses at most one step.

In [ ]:
# 1. GPU check
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb > 35, f"Only {gb:.0f} GB of GPU memory: pick an A100 (40 GB) runtime"

In [ ]:
# 2. Code from Drive
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile
BUNDLE = "/content/drive/MyDrive/geomap/geomap_colab.zip"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
os.makedirs("/content/geomap", exist_ok=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
os.chdir("/content/geomap")
print(sorted(os.listdir(".")))

In [ ]:
# 3. Dependencies (torch comes with Colab; the code runs from src/ without installing the package)
!pip -q install "transformers==4.57.3" nnsight accelerate
import os, sys
os.environ["PYTHONPATH"] = "/content/geomap/src"
sys.path.insert(0, "/content/geomap/src")

In [ ]:
# 4. Hugging Face login (Gemma is gated)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("logged in as", whoami(token=os.environ["HF_TOKEN"])["name"])

In [ ]:
# helper: copy results to Drive after each step
import shutil
def save_to_drive():
    shutil.make_archive("/content/p5_results", "zip", "/content/geomap", "results")
    shutil.copy("/content/p5_results.zip", "/content/drive/MyDrive/geomap/p5_results.zip")
    print("saved to MyDrive/geomap/p5_results.zip")

In [ ]:
# 5. P5a + P5b: natural encoding and write-in (~30-45 min)
!cd /content/geomap && python scripts/p5_encoding.py --model gemma3-12b 2>&1 | grep -v -i warning
save_to_drive()

In [ ]:
# 6. P5c: erase the map (~45-75 min)
!cd /content/geomap && python scripts/p5_erase.py --model gemma3-12b 2>&1 | grep -v -i warning
save_to_drive()

In [ ]:
# 7. Download the results zip to this computer too
from google.colab import files
files.download("/content/p5_results.zip")